In [18]:
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_rows', None)

DATA_DIR = "datasets"  # folder with your 39 CSVs
files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv")]
print(f"Found {len(files)} files")

# First pass: just look at shape and columns of every file
inventory = []
for f in files:
    path = os.path.join(DATA_DIR, f)
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin1")
    try:
        inventory.append({
            "file": f,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "columns": list(df.columns)
        })
    except Exception as e:
        print(f"Failed to read {f}: {e}")

inv_df = pd.DataFrame(inventory)
pd.set_option('display.max_colwidth', None)
print(inv_df[["file", "rows", "cols"]])

Found 40 files
                                   file  rows  cols
0          RS_Session_254_AU_1496.A.csv     7     3
1        RS_Session_254_AU_2384.C.i.csv    27     4
2           RS_Session_254_AU_697_1.csv    37     3
3          RS_Session_255_AU_2349_2.csv    35     3
4           RS_Session_255_AU_305.A.csv     2     4
5           RS_Session_255_AU_749.C.csv     3     3
6           RS_Session_256_AS_154.A.csv     3     3
7          RS_Session_256_AU_2673_1.csv    16     3
8      RS_Session_256_AU_2673_2.ii_.csv    26     4
9            RS_Session_256_AU_95_C.csv    33    11
10       RS_Session_257_AS_71_A.ii_.csv    14     3
11         RS_Session_258_AU_2037_A.csv     4     4
12          RS_Session_258_AU_429_1.csv    35    18
13          RS_Session_258_AU_429_B.csv     4     3
14           RS_Session_258_AU_91_1.csv    35    18
15         RS_Session_259_AU_2028_A.csv     4     3
16         RS_Session_259_AU_2032_A.csv    11     2
17         RS_Session_259_AU_2474_A.csv     3    

In [16]:
def clean_column_names(df):
    """Standardize headers: strip whitespace, fix typos, lowercase-friendly."""
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={"hoi": "Sl. No."})  # fixes the typo in one file
    return df

def clean_numeric_column(series):
    """Convert messy numeric-looking columns to float, handling NA/approx/commas."""
    return (
        series.astype(str)
        .str.replace("approx.", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"NA": np.nan, "N/A": np.nan, "-": np.nan, "": np.nan})
        .astype(float)
    )

def load_and_clean(filename):
    path = os.path.join(DATA_DIR, filename)
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin1")
    df = clean_column_names(df)
    return df

In [17]:
theme_groups = {
    "ev_registration_trends": [
        "RS_Session_255_AU_749.C.csv",
        "RS_Session_256_AS_154.A.csv",
        "RS_Session_259_AU_2032_A.csv",
        "RS_Session_259_AU_2477_C_and_D.csv",
        "RS_Session_260_AU_1872_A_and_B.csv",
        "RS_Session_263_AU_905_A.csv",
        "RS_Session_265_AU_1355_A_and_B.csv",
    ],
    "state_wise_ev_data": [
        "RS_Session_254_AU_697_1.csv",
        "RS_Session_255_AU_2349_2.csv",
        "RS_Session_256_AU_95_C.csv",
        "RS_Session_258_AU_429_1.csv",
        "RS_Session_258_AU_91_1.csv",
        "RS_Session_259_AU_3475_1.csv",
        "RS_Session_260_AU_2349_A_to_B.csv",
    ],
    "charging_infrastructure": [
        "RS_Session_256_AU_2673_1.csv",
        "RS_Session_256_AU_2673_2.ii_.csv",
        "RS_Session_257_AS_71_A.ii_.csv",
        "RS_Session_259_AU_2474_A.csv",
        "RS_Session_259_AU_2837_A.csv",
        "RS_Session_259_AU_2837_B.csv",
        "RS_Session_265_AU_2151_E.csv",
        "RS_Session_266_AU_2960_B_i.csv",
        "RS_Session_266_AU_2960_C_to_D_i.csv",
        "RS_Session_267_AU_581_C_i.csv",
    ],
    "scheme_budget_and_subsidy": [
        "RS_Session_254_AU_1496.A.csv",
        "RS_Session_258_AU_2037_A.csv",
        "RS_Session_258_AU_429_B.csv",
        "RS_Session_259_AU_2028_A.csv",
        "RS_Session_259_AU_2836_A.csv",
        "RS_Session_259_AU_3477_A_to_D.csv",
        "RS_Session_263_AU_105_A.csv",
        "RS_Session_266_AS_217_4.csv",
        "RS_Session_266_AU_553_A_to_B.csv",
        "RS_Session_267_AU_580_A.csv",
    ],
    "sales_by_category": [
        "RS_Session_263_AU_102_A.csv",
        "RS_Session_263_AU_105_C.csv",
        "RS_Session_260_AS_241_E.csv",
    ],
    "global_comparison": [
        "RS_Session_266_AU_552_D_i.csv",
    ],
    "manufacturer_specific": [
        "RS_Session_255_AU_305.A.csv",
        "RS_Session_254_AU_2384.C.i.csv",
    ],
}

# quick check: did every file get assigned to a theme?
assigned = set(sum(theme_groups.values(), []))
unassigned = set(files) - assigned
print("Unassigned files:", unassigned if unassigned else "None — all files grouped")

Unassigned files: None — all files grouped


In [4]:
def build_registration_trends():
    frames = []
    for f in theme_groups["ev_registration_trends"]:
        df = load_and_clean(f)
        df["source_file"] = f
        frames.append(df)
        print(f"\n{f}:\n{df.head(2)}")  # inspect before deciding how to align columns
    return frames

frames = build_registration_trends()


RS_Session_255_AU_749.C.csv:
  Sl. No.  Year  Number of Electric Vehicles                  source_file
0       1  2019                       161314  RS_Session_255_AU_749.C.csv
1       2  2020                       119648  RS_Session_255_AU_749.C.csv

RS_Session_256_AS_154.A.csv:
   Year  Number of Vehicles  Percentage Change WRT Previous Year  \
0  2019              164852                                  NaN   
1  2020              123528                               -25.07   

                   source_file  
0  RS_Session_256_AS_154.A.csv  
1  RS_Session_256_AS_154.A.csv  

RS_Session_259_AU_2032_A.csv:
   Year  Total Count                   source_file
0  2014         2391  RS_Session_259_AU_2032_A.csv
1  2015         7790  RS_Session_259_AU_2032_A.csv

RS_Session_259_AU_2477_C_and_D.csv:
   Year  Electric Vehicles Registered in a Calendar Year  \
0  2020                                           124026   
1  2021                                           329808   

   Percentag

In [5]:
# --- EV Registration Trends: build simple year-level table ---
simple_files = [
    "RS_Session_255_AU_749.C.csv",
    "RS_Session_256_AS_154.A.csv",
    "RS_Session_259_AU_2032_A.csv",
    "RS_Session_259_AU_2477_C_and_D.csv",
    "RS_Session_263_AU_905_A.csv",
    "RS_Session_265_AU_1355_A_and_B.csv",
]

rename_maps = {
    "RS_Session_255_AU_749.C.csv": {"Year": "year", "Number of Electric Vehicles": "ev_count"},
    "RS_Session_256_AS_154.A.csv": {"Year": "year", "Number of Vehicles": "ev_count",
                                     "Percentage Change WRT Previous Year": "pct_change"},
    "RS_Session_259_AU_2032_A.csv": {"Year": "year", "Total Count": "ev_count"},
    "RS_Session_259_AU_2477_C_and_D.csv": {
        "Year": "year",
        "Electric Vehicles Registered in a Calendar Year": "ev_count",
        "Percentage Increase in registration from the previous year": "pct_change"},
    "RS_Session_263_AU_905_A.csv": {"Calendar Year": "year", "Electric Vehicles Registered": "ev_count"},
    "RS_Session_265_AU_1355_A_and_B.csv": {"Year": "year", "Number of Registered Electric Vehicles": "ev_count"},
}

frames = []
for f in simple_files:
    df = load_and_clean(f)
    df = df.rename(columns=rename_maps[f])
    df["source_file"] = f
    df = df[[c for c in ["year", "ev_count", "pct_change", "source_file"] if c in df.columns]]
    frames.append(df)

master_ev_registration_trends = pd.concat(frames, ignore_index=True, sort=False)
master_ev_registration_trends.info()
master_ev_registration_trends

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   year         30 non-null     object 
 1   ev_count     30 non-null     int64  
 2   source_file  30 non-null     object 
 3   pct_change   4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.1+ KB


,year,ev_count,source_file,pct_change
0,2019,161314,RS_Session_255_AU_749.C.csv,NaN
1,2020,119648,RS_Session_255_AU_749.C.csv,NaN
2,Total,280962,RS_Session_255_AU_749.C.csv,NaN
3,2019,164852,RS_Session_256_AS_154.A.csv,NaN
4,2020,123528,RS_Session_256_AS_154.A.csv,-25.07
5,2021,324840,RS_Session_256_AS_154.A.csv,162.97
6,2014,2391,RS_Session_259_AU_2032_A.csv,NaN
7,2015,7790,RS_Session_259_AU_2032_A.csv,NaN
8,2016,49622,RS_Session_259_AU_2032_A.csv,NaN
9,2017,86720,RS_Session_259_AU_2032_A.csv,NaN


In [6]:
master_ev_registration_trends = master_ev_registration_trends[
    ~master_ev_registration_trends["year"].astype(str).str.contains("Total", case=False, na=False)
].reset_index(drop=True)

master_ev_registration_trends.info()
master_ev_registration_trends

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   year         27 non-null     object 
 1   ev_count     27 non-null     int64  
 2   source_file  27 non-null     object 
 3   pct_change   4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 996.0+ bytes


,year,ev_count,source_file,pct_change
0,2019,161314,RS_Session_255_AU_749.C.csv,NaN
1,2020,119648,RS_Session_255_AU_749.C.csv,NaN
2,2019,164852,RS_Session_256_AS_154.A.csv,NaN
3,2020,123528,RS_Session_256_AS_154.A.csv,-25.07
4,2021,324840,RS_Session_256_AS_154.A.csv,162.97
5,2014,2391,RS_Session_259_AU_2032_A.csv,NaN
6,2015,7790,RS_Session_259_AU_2032_A.csv,NaN
7,2016,49622,RS_Session_259_AU_2032_A.csv,NaN
8,2017,86720,RS_Session_259_AU_2032_A.csv,NaN
9,2018,129125,RS_Session_259_AU_2032_A.csv,NaN


In [7]:
# --- EV Registration Trends: category-wise wide table → long format ---
df_wide = load_and_clean("RS_Session_260_AU_1872_A_and_B.csv")

records = []
years = ["2018", "2019", "2020", "2021", "2022", "2023 (Till 01-08-2023)"]
for _, row in df_wide.iterrows():
    for yr in years:
        records.append({
            "year": yr,
            "vehicle_category": row["Vehicle Category"],
            "total_vehicles": row.get(f"{yr} - Total"),
            "ev_count": row.get(f"{yr} - EV"),
            "ev_pct": row.get(f"{yr} - %"),
            "source_file": "RS_Session_260_AU_1872_A_and_B.csv"
        })

master_ev_registration_by_category = pd.DataFrame(records)
master_ev_registration_by_category.info()
master_ev_registration_by_category

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              30 non-null     object 
 1   vehicle_category  30 non-null     object 
 2   total_vehicles    30 non-null     int64  
 3   ev_count          30 non-null     int64  
 4   ev_pct            30 non-null     float64
 5   source_file       30 non-null     object 
dtypes: float64(1), int64(2), object(3)
memory usage: 1.5+ KB


,year,vehicle_category,total_vehicles,ev_count,ev_pct,source_file
0,2018,Two Wheeler,19576235,17067,0.09,RS_Session_260_AU_1872_A_and_B.csv
1,2019,Two Wheeler,18644700,30389,0.16,RS_Session_260_AU_1872_A_and_B.csv
2,2020,Two Wheeler,14305129,29113,0.20,RS_Session_260_AU_1872_A_and_B.csv
3,2021,Two Wheeler,13926217,156243,1.12,RS_Session_260_AU_1872_A_and_B.csv
4,2022,Two Wheeler,15592118,631181,4.05,RS_Session_260_AU_1872_A_and_B.csv
5,2023 (Till 01-08-2023),Two Wheeler,9276337,489637,5.28,RS_Session_260_AU_1872_A_and_B.csv
6,2018,Three Wheeler,764806,110133,14.40,RS_Session_260_AU_1872_A_and_B.csv
7,2019,Three Wheeler,765867,133489,17.43,RS_Session_260_AU_1872_A_and_B.csv
8,2020,Three Wheeler,400893,90385,22.55,RS_Session_260_AU_1872_A_and_B.csv
9,2021,Three Wheeler,390820,158129,40.46,RS_Session_260_AU_1872_A_and_B.csv


## Insights: EV Registration Trends

- Overall EV registrations show strong, consistent growth from 2019 onward,
  accelerating sharply from 2021–2022 — one series jumps from ~124,000 (2020)
  to ~1,020,000 (2022), roughly 8x in two years.
- A clear dip appears in 2020 across multiple independent files (e.g. one
  series drops from ~165,000 to ~123,000, -25% YoY) — consistent with
  COVID-19 disrupting vehicle sales broadly that year.
- Category-wise adoption is very uneven, not a single uniform trend:
  - Three-wheelers lead by far — EV share rose from 14.4% (2018) to over
    52% (2023), likely driven by commercial/gig use (e-rickshaws, delivery).
  - Two-wheelers grew from ~0.1% to ~5.3% — real but much slower growth.
  - Four-wheelers (passenger cars) stayed under 2% EV share even by 2023 —
    adoption here is still in a very early stage.
  - Goods vehicles and public service vehicles remain under 1–4%.
- Takeaway: "EV adoption in India" is really a three-wheeler-led transition
  first, with two-wheelers following at a distance, and passenger cars
  barely moving yet.
- Caveat: these are separate government disclosures answering separate
  Parliament questions, not one continuous time series — cross-file
  comparisons (like the 2020 dip) are suggestive, not statistically
  confirmed, until formally joined during EDA.

In [9]:
for f in theme_groups["state_wise_ev_data"]:
    df = load_and_clean(f)
    print(f"\n{f} — shape {df.shape}")
    display(df.head(2))


RS_Session_254_AU_697_1.csv — shape (37, 3)


,Sl. No.,State/UT,Total Number of Invoice/Sales
0,1,Jammu Kashmir,437
1,2,Himachal Pradesh,241



RS_Session_255_AU_2349_2.csv — shape (35, 3)


,Sl. No.,State/UT,Total Number of Invoices/Sales
0,1,Jammu Kashmir,1036
1,2,Himachal Pradesh,446



RS_Session_256_AU_95_C.csv — shape (33, 11)


...

,Sr. No.,State/UT Name,2WN,2WT,2WIC,3WN,3WT,LMV,LPV,LGV,4WIC,MMV,MPV,MGV,HPV,HGV,OTH,Grand Total
0,1,Andaman and Nicobar Island,2,5.0,NaN,NaN,30.0,86,6.0,NaN,NaN,NaN,NaN,NaN,40.0,NaN,NaN,169
1,2,Andhra Pradesh,27629,NaN,2.0,374.0,108.0,1050,3.0,166.0,NaN,NaN,NaN,NaN,NaN,NaN,1117.0,30449



RS_Session_259_AU_3475_1.csv — shape (35, 3)


,S. No.,State Name,Electric Vehicle Count
0,1,Andaman and Nicobar Island,182
1,2,Andhra Pradesh,51322



RS_Session_260_AU_2349_A_to_B.csv — shape (35, 4)


,Sl.No.,State/UT,Electric,Non-electric
0,1,Andaman and Nicobar Islands,190,161258
1,2,Andhra Pradesh,67905,16553509


In [10]:
df_429 = load_and_clean("RS_Session_258_AU_429_1.csv")
df_91 = load_and_clean("RS_Session_258_AU_91_1.csv")
print(df_429.drop(columns=["Sl.No." if "Sl.No." in df_429.columns else "S.No."], errors="ignore").equals(
    df_91.drop(columns=["Sr. No."], errors="ignore")
))

False


In [11]:
df_429 = load_and_clean("RS_Session_258_AU_429_1.csv")
df_91 = load_and_clean("RS_Session_258_AU_91_1.csv")

# align state name columns so we can compare row by row
state_col_429 = "State Name"
state_col_91 = "State/UT Name"

df_429_sorted = df_429.sort_values(state_col_429).reset_index(drop=True)
df_91_sorted = df_91.sort_values(state_col_91).reset_index(drop=True)

print("Shapes:", df_429_sorted.shape, df_91_sorted.shape)
print("\nStates in 429 but not in 91:",
      set(df_429_sorted[state_col_429]) - set(df_91_sorted[state_col_91]))
print("States in 91 but not in 429:",
      set(df_91_sorted[state_col_91]) - set(df_429_sorted[state_col_429]))

# compare Grand Total column state by state
compare = df_429_sorted[[state_col_429, "Grand Total"]].merge(
    df_91_sorted[[state_col_91, "Grand Total"]],
    left_on=state_col_429, right_on=state_col_91,
    suffixes=("_429", "_91")
)
compare["diff"] = compare["Grand Total_429"] - compare["Grand Total_91"]
compare[compare["diff"] != 0]

Shapes: (35, 18) (35, 18)

States in 429 but not in 91: set()
States in 91 but not in 429: set()


,State Name,Grand Total_429,State/UT Name,Grand Total_91,diff


In [22]:
# --- Group 3: vehicle type-code breakdown by state (melt wide -> long) ---
# Note: RS_Session_258_AU_429_1.csv is a duplicate of this file (verified identical
# Grand Totals for all 35 states) — excluded to avoid double-counting.

df_type = load_and_clean("RS_Session_258_AU_91_1.csv")
df_type = df_type[~df_type["State/UT Name"].astype(str).str.contains("Total", case=False, na=False)]

type_cols = [c for c in df_type.columns if c not in
             ["Sr. No.", "State/UT Name", "Grand Total", "source_file"]]

master_state_vehicle_type = df_type.melt(
    id_vars=["State/UT Name"],
    value_vars=type_cols,
    var_name="vehicle_type_code",
    value_name="count"
).rename(columns={"State/UT Name": "state"})
master_state_vehicle_type["source_file"] = "RS_Session_258_AU_91_1.csv"

master_state_vehicle_type.info()
master_state_vehicle_type

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   state              510 non-null    object 
 1   vehicle_type_code  510 non-null    object 
 2   count              276 non-null    float64
 3   source_file        510 non-null    object 
dtypes: float64(1), object(3)
memory usage: 16.1+ KB


,state,vehicle_type_code,count,source_file
0,Andaman and Nicobar Island,2WN,2.0,RS_Session_258_AU_91_1.csv
1,Andhra Pradesh,2WN,27629.0,RS_Session_258_AU_91_1.csv
2,Arunachal Pradesh,2WN,14.0,RS_Session_258_AU_91_1.csv
3,Assam,2WN,2287.0,RS_Session_258_AU_91_1.csv
4,Bihar,2WN,13472.0,RS_Session_258_AU_91_1.csv
5,Chandigarh,2WN,1004.0,RS_Session_258_AU_91_1.csv
6,Chhattisgarh,2WN,20112.0,RS_Session_258_AU_91_1.csv
7,Delhi,2WN,43627.0,RS_Session_258_AU_91_1.csv
8,Goa,2WN,5555.0,RS_Session_258_AU_91_1.csv
9,Gujarat,2WN,67990.0,RS_Session_258_AU_91_1.csv


In [21]:
group1_files = {
    "RS_Session_254_AU_697_1.csv": "Total Number of Invoice/Sales",
    "RS_Session_255_AU_2349_2.csv": "Total Number of Invoices/Sales",
    "RS_Session_259_AU_3475_1.csv": "Electric Vehicle Count",
}

frames = []
for f, value_col in group1_files.items():
    df = load_and_clean(f)
    state_col = "State/UT" if "State/UT" in df.columns else "State Name"
    df = df.rename(columns={state_col: "state", value_col: "count"})
    df["metric"] = value_col
    df["source_file"] = f
    df = df[["state", "count", "metric", "source_file"]]
    frames.append(df)

master_state_simple_counts = pd.concat(frames, ignore_index=True)
master_state_simple_counts = master_state_simple_counts[
    ~master_state_simple_counts["state"].astype(str).str.contains("Total", case=False, na=False)
].reset_index(drop=True)

master_state_simple_counts.info()
master_state_simple_counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   state        104 non-null    object
 1   count        104 non-null    int64 
 2   metric       104 non-null    object
 3   source_file  104 non-null    object
dtypes: int64(1), object(3)
memory usage: 3.4+ KB


,state,count,metric,source_file
0,Jammu Kashmir,437,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
1,Himachal Pradesh,241,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
2,Punjab,764,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
3,Chandigarh,48,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
4,Uttarakhand,1057,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
5,Haryana,1477,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
6,Delhi,6413,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
7,Rajasthan,6721,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
8,Uttar Pradesh,6022,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv
9,Bihar,2615,Total Number of Invoice/Sales,RS_Session_254_AU_697_1.csv


In [20]:
df_cat = load_and_clean("RS_Session_256_AU_95_C.csv")
df_cat = df_cat[~df_cat["State Name"].astype(str).str.contains("Total", case=False, na=False)]

category_cols = [c for c in df_cat.columns if c not in ["State Name", "Grand Total"]]

master_state_vehicle_category = df_cat.melt(
    id_vars=["State Name"],
    value_vars=category_cols,
    var_name="vehicle_category",
    value_name="count"
).rename(columns={"State Name": "state"})
master_state_vehicle_category["source_file"] = "RS_Session_256_AU_95_C.csv"

master_state_vehicle_category.info()
master_state_vehicle_category

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   state             288 non-null    object 
 1   vehicle_category  288 non-null    object 
 2   count             191 non-null    float64
 3   source_file       288 non-null    object 
dtypes: float64(1), object(3)
memory usage: 9.1+ KB


,state,vehicle_category,count,source_file
0,Andaman and Nicobar Island,Two Wheeler,1.0,RS_Session_256_AU_95_C.csv
1,Arunachal Pradesh,Two Wheeler,14.0,RS_Session_256_AU_95_C.csv
2,Assam,Two Wheeler,721.0,RS_Session_256_AU_95_C.csv
3,Bihar,Two Wheeler,5003.0,RS_Session_256_AU_95_C.csv
4,Chandigarh,Two Wheeler,298.0,RS_Session_256_AU_95_C.csv
5,Chhattisgarh,Two Wheeler,6424.0,RS_Session_256_AU_95_C.csv
6,Delhi,Two Wheeler,14730.0,RS_Session_256_AU_95_C.csv
7,Goa,Two Wheeler,1314.0,RS_Session_256_AU_95_C.csv
8,Gujarat,Two Wheeler,13662.0,RS_Session_256_AU_95_C.csv
9,Haryana,Two Wheeler,7777.0,RS_Session_256_AU_95_C.csv


In [19]:
df_ev_split = load_and_clean("RS_Session_260_AU_2349_A_to_B.csv")
df_ev_split = df_ev_split[~df_ev_split["State/UT"].astype(str).str.contains("Total", case=False, na=False)]

master_state_electric_split = df_ev_split.melt(
    id_vars=["State/UT"],
    value_vars=["Electric", "Non-electric"],
    var_name="category",
    value_name="count"
).rename(columns={"State/UT": "state"})
master_state_electric_split["source_file"] = "RS_Session_260_AU_2349_A_to_B.csv"

master_state_electric_split.info()
master_state_electric_split

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   state        68 non-null     object
 1   category     68 non-null     object
 2   count        68 non-null     int64 
 3   source_file  68 non-null     object
dtypes: int64(1), object(3)
memory usage: 2.3+ KB


,state,category,count,source_file
0,Andaman and Nicobar Islands,Electric,190,RS_Session_260_AU_2349_A_to_B.csv
1,Andhra Pradesh,Electric,67905,RS_Session_260_AU_2349_A_to_B.csv
2,Arunachal Pradesh,Electric,28,RS_Session_260_AU_2349_A_to_B.csv
3,Assam,Electric,120423,RS_Session_260_AU_2349_A_to_B.csv
4,Bihar,Electric,161060,RS_Session_260_AU_2349_A_to_B.csv
5,Chandigarh,Electric,7964,RS_Session_260_AU_2349_A_to_B.csv
6,Chhattisgarh,Electric,54848,RS_Session_260_AU_2349_A_to_B.csv
7,Delhi,Electric,233212,RS_Session_260_AU_2349_A_to_B.csv
8,Goa,Electric,12615,RS_Session_260_AU_2349_A_to_B.csv
9,Gujarat,Electric,138410,RS_Session_260_AU_2349_A_to_B.csv


## Insights: State-wise EV Data

- EV adoption is highly concentrated geographically, not evenly spread —
  large states like Uttar Pradesh show EV counts in the hundreds of
  thousands, while smaller states/UTs (Andaman & Nicobar, Arunachal
  Pradesh) have counts in the dozens or low hundreds — several orders of
  magnitude apart.
- Even in "leading" states, EVs remain a tiny share of the total vehicle
  fleet — e.g. one state's electric vs. non-electric split showed under
  0.5% electrification when measured against total registered vehicles,
  not just against other EVs.
- Data-quality finding: two files with different filenames/session numbers
  (RS_Session_258_AU_429_1.csv and RS_Session_258_AU_91_1.csv) turned out
  to contain identical state-level totals — a reminder that government
  open-data releases can duplicate/overlap, and blind merging without a
  verification step would have silently doubled every state's numbers.
- Several category/type-code breakdowns have missing values (NaN) for some
  states — likely representing zero vehicles in that category rather than
  missing data, since the original wide tables just left those cells
  blank. To be confirmed and converted during analysis, not assumed here.

In [23]:
for f in theme_groups["charging_infrastructure"]:
    df = load_and_clean(f)
    print(f"\n{f} — shape {df.shape}")
    display(df.head(2))


RS_Session_256_AU_2673_1.csv — shape (16, 3)


,Category,City/Highway,Charging Stations
0,City,Chandigarh,48
1,City,Delhi,94



RS_Session_256_AU_2673_2.ii_.csv — shape (26, 4)


,Sl. No,Category,Expressways/Highways,EV Charging Stations Sanctioned
0,1,Expressways,Mumbai - Pune,10
1,2,Expressways,Ahmadabad - Vadodara,10



RS_Session_257_AS_71_A.ii_.csv — shape (14, 3)


...

,Sl. No.,State/UT,No. of EVPCS
0,1,Karnataka,5765
1,2,Maharashtra,3728



RS_Session_266_AU_2960_C_to_D_i.csv — shape (32, 3)


,Sl. No.,District,No. of EVPCS
0,1,Raipur,57
1,2,Bilaspur,28



RS_Session_267_AU_581_C_i.csv — shape (37, 5)


,Sl. No.,State/UT,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charging Stations energized as on 01-01-2025,Total No. of EV charging stations installed by OMCs from their own funds as on 01-01-2025
0,1,Andaman and Nicobar Islands,0,0,6
1,2,Andhra Pradesh,354,20,912


In [24]:
group_a_files = {
    "RS_Session_257_AS_71_A.ii_.csv": "Operational Charging Stations Under FAME-I",
    "RS_Session_259_AU_2837_A.csv": "No. of Operational PCS",
    "RS_Session_265_AU_2151_E.csv": "No. of PCS as on 31st March 2024",
    "RS_Session_266_AU_2960_B_i.csv": "No. of EVPCS",
}

frames = []
for f, value_col in group_a_files.items():
    df = load_and_clean(f)
    state_col = "State/UT" if "State/UT" in df.columns else "State/ UT"
    df = df.rename(columns={state_col: "state", value_col: "count"})
    df["metric"] = value_col
    df["source_file"] = f
    frames.append(df[["state", "count", "metric", "source_file"]])

master_charging_state_simple = pd.concat(frames, ignore_index=True)
master_charging_state_simple = master_charging_state_simple[
    ~master_charging_state_simple["state"].astype(str).str.contains("Total", case=False, na=False)
].reset_index(drop=True)

master_charging_state_simple.info()
master_charging_state_simple

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   state        117 non-null    object
 1   count        117 non-null    int64 
 2   metric       117 non-null    object
 3   source_file  117 non-null    object
dtypes: int64(1), object(3)
memory usage: 3.8+ KB


,state,count,metric,source_file
0,Telangana,57,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
1,Jharkhand,30,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
2,Goa,30,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
3,Karnataka,65,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
4,Himachal Pradesh,9,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
5,Uttar Pradesh,16,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
6,Rajasthan,49,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
7,Delhi,94,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
8,Chandigarh UT,48,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv
9,Delhi-Jaipur- Agra Highway,31,Operational Charging Stations Under FAME-I,RS_Session_257_AS_71_A.ii_.csv


In [25]:
df_fame2 = load_and_clean("RS_Session_267_AU_581_C_i.csv")
df_fame2 = df_fame2[~df_fame2["State/UT"].astype(str).str.contains("Total", case=False, na=False)]

value_cols = [c for c in df_fame2.columns if c not in ["Sl. No.", "State/UT"]]

master_charging_fame2_detail = df_fame2.melt(
    id_vars=["State/UT"],
    value_vars=value_cols,
    var_name="metric",
    value_name="count"
).rename(columns={"State/UT": "state"})
master_charging_fame2_detail["source_file"] = "RS_Session_267_AU_581_C_i.csv"

master_charging_fame2_detail.info()
master_charging_fame2_detail

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   state        108 non-null    object
 1   metric       108 non-null    object
 2   count        108 non-null    int64 
 3   source_file  108 non-null    object
dtypes: int64(1), object(3)
memory usage: 3.5+ KB


,state,metric,count,source_file
0,Andaman and Nicobar Islands,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,0,RS_Session_267_AU_581_C_i.csv
1,Andhra Pradesh,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,354,RS_Session_267_AU_581_C_i.csv
2,Arunachal Pradesh,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,2,RS_Session_267_AU_581_C_i.csv
3,Assam,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,83,RS_Session_267_AU_581_C_i.csv
4,Bihar,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,58,RS_Session_267_AU_581_C_i.csv
5,Chandigarh,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,0,RS_Session_267_AU_581_C_i.csv
6,Chhattisgarh,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,30,RS_Session_267_AU_581_C_i.csv
7,Delhi,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,41,RS_Session_267_AU_581_C_i.csv
8,Goa,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,9,RS_Session_267_AU_581_C_i.csv
9,Gujarat,EV Charging Stations under FAME-II Subsidy Scheme - No. of EV Charger installed as on 01-01-2025,312,RS_Session_267_AU_581_C_i.csv


In [26]:
df_city = load_and_clean("RS_Session_256_AU_2673_1.csv")
df_city = df_city.rename(columns={"City/Highway": "location", "Charging Stations": "count"})
df_city["source_file"] = "RS_Session_256_AU_2673_1.csv"

master_charging_city_highway = df_city[~df_city["location"].astype(str).str.contains("Total", case=False, na=False)]
master_charging_city_highway.info()
master_charging_city_highway

<class 'pandas.core.frame.DataFrame'>
Index: 14 entries, 0 to 14
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Category     14 non-null     object
 1   location     14 non-null     object
 2   count        14 non-null     int64 
 3   source_file  14 non-null     object
dtypes: int64(1), object(3)
memory usage: 560.0+ bytes


,Category,location,count,source_file
0,City,Chandigarh,48,RS_Session_256_AU_2673_1.csv
1,City,Delhi,94,RS_Session_256_AU_2673_1.csv
2,City,Jaipur,49,RS_Session_256_AU_2673_1.csv
3,City,B'Lore,60,RS_Session_256_AU_2673_1.csv
4,City,Ranchi,30,RS_Session_256_AU_2673_1.csv
5,City,Lucknow,1,RS_Session_256_AU_2673_1.csv
6,City,Goa,30,RS_Session_256_AU_2673_1.csv
7,City,Hyderabad,57,RS_Session_256_AU_2673_1.csv
8,City,Agra,15,RS_Session_256_AU_2673_1.csv
9,City,Shimla,9,RS_Session_256_AU_2673_1.csv


In [27]:
df_exp = load_and_clean("RS_Session_256_AU_2673_2.ii_.csv")
df_exp = df_exp.rename(columns={"Expressways/Highways": "route", "EV Charging Stations Sanctioned": "count"})
df_exp["source_file"] = "RS_Session_256_AU_2673_2.ii_.csv"

master_charging_expressway = df_exp[~df_exp["route"].astype(str).str.contains("Total", case=False, na=False)]
master_charging_expressway.info()
master_charging_expressway

<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Sl. No       25 non-null     object
 1   Category     25 non-null     object
 2   route        25 non-null     object
 3   count        25 non-null     int64 
 4   source_file  25 non-null     object
dtypes: int64(1), object(4)
memory usage: 1.2+ KB


,Sl. No,Category,route,count,source_file
0,1,Expressways,Mumbai - Pune,10,RS_Session_256_AU_2673_2.ii_.csv
1,2,Expressways,Ahmadabad - Vadodara,10,RS_Session_256_AU_2673_2.ii_.csv
2,3,Expressways,Delhi Agra Yamuna,20,RS_Session_256_AU_2673_2.ii_.csv
3,4,Expressways,Bengaluru Mysore,14,RS_Session_256_AU_2673_2.ii_.csv
4,5,Expressways,Bangaluru-Chennai,30,RS_Session_256_AU_2673_2.ii_.csv
5,6,Expressways,Surat-Mumbai,30,RS_Session_256_AU_2673_2.ii_.csv
6,7,Expressways,Agra-Lucknow,40,RS_Session_256_AU_2673_2.ii_.csv
7,8,Expressways,Eastern Peripheral (A),14,RS_Session_256_AU_2673_2.ii_.csv
8,9,Expressways,Hyderabad ORR,16,RS_Session_256_AU_2673_2.ii_.csv
9,1,Highways,Delhi - Srinagar,80,RS_Session_256_AU_2673_2.ii_.csv


In [28]:
df_org = load_and_clean("RS_Session_259_AU_2474_A.csv")
df_org["source_file"] = "RS_Session_259_AU_2474_A.csv"
master_charging_by_organization = df_org
master_charging_by_organization.info()
master_charging_by_organization

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Charging Stations details  3 non-null      object
 1   Organization               3 non-null      object
 2   Sanctioned                 3 non-null      int64 
 3   Installed                  3 non-null      int64 
 4   source_file                3 non-null      object
dtypes: int64(2), object(3)
memory usage: 252.0+ bytes


,Charging Stations details,Organization,Sanctioned,Installed,source_file
0,"Solar Based Charging Infrastructure for EVs in NCR by REIL, Jaipur","REIL, Jaipur",3,3,RS_Session_259_AU_2474_A.csv
1,Solar Grid Hybrid and Grid powered Charging Station along Delhi-Jaipur-Agra Highway,REIL,25,25,RS_Session_259_AU_2474_A.csv
2,Solar Based Chargers (20 locations) along Delhi Chandigarh Highway,BHEL,20,20,RS_Session_259_AU_2474_A.csv


In [29]:
df_nh = load_and_clean("RS_Session_259_AU_2837_B.csv")
df_nh = df_nh.rename(columns={"National Highway": "highway", "No. of Operational PCS": "count"})
df_nh["source_file"] = "RS_Session_259_AU_2837_B.csv"

master_charging_national_highway = df_nh[~df_nh["highway"].astype(str).str.contains("Total", case=False, na=False)]
master_charging_national_highway.info()
master_charging_national_highway

<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 0 to 89
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   highway      90 non-null     object
 1   count        90 non-null     int64 
 2   source_file  90 non-null     object
dtypes: int64(1), object(2)
memory usage: 2.8+ KB


,highway,count,source_file
0,National Highway-10,1,RS_Session_259_AU_2837_B.csv
1,National Highway-11,3,RS_Session_259_AU_2837_B.csv
2,National Highway-128,1,RS_Session_259_AU_2837_B.csv
3,National Highway-13,3,RS_Session_259_AU_2837_B.csv
4,National Highway-130,1,RS_Session_259_AU_2837_B.csv
5,National Highway-130A,2,RS_Session_259_AU_2837_B.csv
6,National Highway-130B,2,RS_Session_259_AU_2837_B.csv
7,National Highway-130C,2,RS_Session_259_AU_2837_B.csv
8,National Highway-135,2,RS_Session_259_AU_2837_B.csv
9,National Highway-143,2,RS_Session_259_AU_2837_B.csv


In [30]:
df_dist = load_and_clean("RS_Session_266_AU_2960_C_to_D_i.csv")
df_dist = df_dist.rename(columns={"No. of EVPCS": "count"})
df_dist["source_file"] = "RS_Session_266_AU_2960_C_to_D_i.csv"

master_charging_district = df_dist[~df_dist["District"].astype(str).str.contains("Total", case=False, na=False)]
master_charging_district.info()
master_charging_district

<class 'pandas.core.frame.DataFrame'>
Index: 31 entries, 0 to 30
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Sl. No.      31 non-null     object
 1   District     31 non-null     object
 2   count        31 non-null     int64 
 3   source_file  31 non-null     object
dtypes: int64(1), object(3)
memory usage: 1.2+ KB


,Sl. No.,District,count,source_file
0,1,Raipur,57,RS_Session_266_AU_2960_C_to_D_i.csv
1,2,Bilaspur,28,RS_Session_266_AU_2960_C_to_D_i.csv
2,3,Rajnandgaon,24,RS_Session_266_AU_2960_C_to_D_i.csv
3,4,Durg,19,RS_Session_266_AU_2960_C_to_D_i.csv
4,5,Surguja,15,RS_Session_266_AU_2960_C_to_D_i.csv
5,6,Janjgir-Champa,13,RS_Session_266_AU_2960_C_to_D_i.csv
6,7,Korba,11,RS_Session_266_AU_2960_C_to_D_i.csv
7,8,Mahasamund,10,RS_Session_266_AU_2960_C_to_D_i.csv
8,9,Balod,9,RS_Session_266_AU_2960_C_to_D_i.csv
9,10,Dhamtari,9,RS_Session_266_AU_2960_C_to_D_i.csv


## Insights: Charging Infrastructure

- Charging stations are extremely concentrated in a handful of states —
  Karnataka (5,765 EVPCS) and Maharashtra (3,728) dominate, while many
  smaller states/UTs have single digits.
- Delhi stands out for *operational* public charging stations specifically
  (1,886+) even though it's not top of the EVPCS list — worth checking why
  in analysis (different metric/date, or genuinely different infra type).
- OMC (oil company)-funded stations are a much bigger number than
  FAME-II subsidized ones in most states (e.g. UP: 2,561 OMC-funded vs.
  269 FAME-II installed) — suggests private/OMC investment is outpacing
  government subsidy rollout.
- FAME-I data mixes actual states with named highway corridors in the same
  column — needs separating before any state-level comparison.
- Two files (429_1 and 91_1) turned out to be exact duplicates — a reminder
  that government data releases can overlap.
  

In [31]:
for f in theme_groups["scheme_budget_and_subsidy"]:
    df = load_and_clean(f)
    print(f"\n{f} — shape {df.shape}")
    display(df.head(2))


RS_Session_254_AU_1496.A.csv — shape (7, 3)


,Financial Year,Fund Allocation,Fund Utilization
0,2015-16,75.0,75.0
1,2016-17,144.0,144.0



RS_Session_258_AU_2037_A.csv — shape (4, 4)


,Sl. No.,Financial Year,Budget Allocation,Fund Utilization as on 30.11.2022
0,1,2019-2020,500.00,500.00
1,2,2020-2021,318.36,318.36



RS_Session_258_AU_429_B.csv — shape (4, 3)


...

,Years,Energy Requirement
0,2024-25,7170
1,2025-26,12160



RS_Session_266_AU_553_A_to_B.csv — shape (6, 3)


,Sl. No.,Electric Vehicle segment supported under PM E-DRIVE Scheme,Budget Allocated for the vehicle segment
0,1,Registered e-2Wheelers,1772
1,2,Registered e-3 Wheelers - e-Rickshaws and e-Cart,192



RS_Session_267_AU_580_A.csv — shape (4, 3)


,Sl. No.,EV Segment,Total No. of EVs supported
0,1,2 wheeler,1428009
1,2,3 wheeler,164180


In [32]:
df_2836 = load_and_clean("RS_Session_259_AU_2836_A.csv")
df_3477 = load_and_clean("RS_Session_259_AU_3477_A_to_D.csv")
print(df_2836.equals(df_3477))
print(df_2836)
print(df_3477)


False
  Category  Vehicles to be Supported (in Nos)  \
0     e-2W                            1000000   
1     e-3W                             500000   
2     e-4W                              55000   
3  e-Buses                               7090   
4    Total                            1562090   

   Actual Vehicles supported (in Nos)  
0                              792529  
1                               81172  
2                                6831  
3                                2435  
4                              882967  
  Category  Vehicles to be Supported (in No.)  \
0     e-2W                            1000000   
1     e-3W                             500000   
2     e-4W                              55000   
3  e-Buses                               7090   
4    Total                            1562090   

   Actual Vehicles Supported (in No.)  
0                              792529  
1                               81172  
2                                6831  
3   

In [34]:
group_a_files = [
    "RS_Session_254_AU_1496.A.csv",
    "RS_Session_258_AU_2037_A.csv",
    "RS_Session_259_AU_2028_A.csv",
    "RS_Session_263_AU_105_A.csv",
]

frames = []
for f in group_a_files:
    df = load_and_clean(f)
    util_col = [c for c in df.columns if "Fund Utilization" in c][0]
    alloc_col = "Budget Allocation" if "Budget Allocation" in df.columns else "Fund Allocation"
    df = df.rename(columns={"Financial Year": "financial_year",
                             alloc_col: "budget_allocation",
                             util_col: "fund_utilization"})
    df["source_file"] = f
    frames.append(df[["financial_year", "budget_allocation", "fund_utilization", "source_file"]])

master_scheme_budget_by_year = pd.concat(frames, ignore_index=True, sort=False)
master_scheme_budget_by_year.info()
master_scheme_budget_by_year

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   financial_year     20 non-null     object 
 1   budget_allocation  20 non-null     float64
 2   fund_utilization   20 non-null     float64
 3   source_file        20 non-null     object 
dtypes: float64(2), object(2)
memory usage: 772.0+ bytes


,financial_year,budget_allocation,fund_utilization,source_file
0,2015-16,75.00,75.00,RS_Session_254_AU_1496.A.csv
1,2016-17,144.00,144.00,RS_Session_254_AU_1496.A.csv
2,2017-18,165.00,165.00,RS_Session_254_AU_1496.A.csv
3,2018-19,145.00,145.00,RS_Session_254_AU_1496.A.csv
4,2019-20,500.00,500.00,RS_Session_254_AU_1496.A.csv
5,2020-21,318.36,318.36,RS_Session_254_AU_1496.A.csv
6,2021-22,756.66,53.27,RS_Session_254_AU_1496.A.csv
7,2019-2020,500.00,500.00,RS_Session_258_AU_2037_A.csv
8,2020-2021,318.36,318.36,RS_Session_258_AU_2037_A.csv
9,2021-2022,800.00,800.00,RS_Session_258_AU_2037_A.csv


In [35]:
df_amount = load_and_clean("RS_Session_258_AU_429_B.csv")
df_amount = df_amount.rename(columns={"Amount (in Rs. Crore)": "amount_rs_crore"})
df_amount["amount_rs_crore"] = clean_numeric_column(df_amount["amount_rs_crore"])
df_amount["source_file"] = "RS_Session_258_AU_429_B.csv"

master_scheme_subsidy_amount = df_amount
master_scheme_subsidy_amount.info()
master_scheme_subsidy_amount

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   S.No.            4 non-null      int64  
 1   Category         4 non-null      object 
 2   amount_rs_crore  4 non-null      float64
 3   source_file      4 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 260.0+ bytes


,S.No.,Category,amount_rs_crore,source_file
0,1,e-2 Wheelers,2464.27,RS_Session_258_AU_429_B.csv
1,2,e-3 Wheelers,351.21,RS_Session_258_AU_429_B.csv
2,3,e-4 Wheelers,114.65,RS_Session_258_AU_429_B.csv
3,4,e-buses,687.93,RS_Session_258_AU_429_B.csv


In [36]:
df_targets = load_and_clean("RS_Session_259_AU_2836_A.csv")
df_targets = df_targets[~df_targets["Category"].astype(str).str.contains("Total", case=False, na=False)]
df_targets = df_targets.rename(columns={
    "Vehicles to be Supported (in Nos)": "target_vehicles",
    "Actual Vehicles supported (in Nos)": "actual_vehicles"
})
df_targets["source_file"] = "RS_Session_259_AU_2836_A.csv"
# note: RS_Session_259_AU_3477_A_to_D.csv is a duplicate of this file — excluded

master_scheme_vehicle_targets = df_targets
master_scheme_vehicle_targets.info()
master_scheme_vehicle_targets

<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Category         4 non-null      object
 1   target_vehicles  4 non-null      int64 
 2   actual_vehicles  4 non-null      int64 
 3   source_file      4 non-null      object
dtypes: int64(2), object(2)
memory usage: 160.0+ bytes


,Category,target_vehicles,actual_vehicles,source_file
0,e-2W,1000000,792529,RS_Session_259_AU_2836_A.csv
1,e-3W,500000,81172,RS_Session_259_AU_2836_A.csv
2,e-4W,55000,6831,RS_Session_259_AU_2836_A.csv
3,e-Buses,7090,2435,RS_Session_259_AU_2836_A.csv


In [37]:
df_energy = load_and_clean("RS_Session_266_AS_217_4.csv")
df_energy = df_energy.rename(columns={"Years": "year"})
df_energy["source_file"] = "RS_Session_266_AS_217_4.csv"

master_scheme_energy_requirement = df_energy
master_scheme_energy_requirement.info()
master_scheme_energy_requirement

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   year                6 non-null      object
 1   Energy Requirement  6 non-null      int64 
 2   source_file         6 non-null      object
dtypes: int64(1), object(2)
memory usage: 276.0+ bytes


,year,Energy Requirement,source_file
0,2024-25,7170,RS_Session_266_AS_217_4.csv
1,2025-26,12160,RS_Session_266_AS_217_4.csv
2,2026-27,18910,RS_Session_266_AS_217_4.csv
3,2027-28,27688,RS_Session_266_AS_217_4.csv
4,2028-29,38788,RS_Session_266_AS_217_4.csv
5,2029-30,52604,RS_Session_266_AS_217_4.csv


In [38]:
df_pmedrive = load_and_clean("RS_Session_266_AU_553_A_to_B.csv")
df_pmedrive = df_pmedrive.rename(columns={
    "Electric Vehicle segment supported under PM E-DRIVE Scheme": "segment",
    "Budget Allocated for the vehicle segment": "budget_allocated"
})
df_pmedrive["source_file"] = "RS_Session_266_AU_553_A_to_B.csv"

master_scheme_pmedrive_budget = df_pmedrive[["segment", "budget_allocated", "source_file"]]
master_scheme_pmedrive_budget.info()
master_scheme_pmedrive_budget

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   segment           6 non-null      object
 1   budget_allocated  6 non-null      int64 
 2   source_file       6 non-null      object
dtypes: int64(1), object(2)
memory usage: 276.0+ bytes


,segment,budget_allocated,source_file
0,Registered e-2Wheelers,1772,RS_Session_266_AU_553_A_to_B.csv
1,Registered e-3 Wheelers - e-Rickshaws and e-Cart,192,RS_Session_266_AU_553_A_to_B.csv
2,Registered e-3Wheelers (L5),715,RS_Session_266_AU_553_A_to_B.csv
3,e-ambulance,500,RS_Session_266_AU_553_A_to_B.csv
4,e-trucks and other emerging EVs,500,RS_Session_266_AU_553_A_to_B.csv
5,Registered e-buses,4391,RS_Session_266_AU_553_A_to_B.csv


In [39]:
df_supported = load_and_clean("RS_Session_267_AU_580_A.csv")
df_supported = df_supported.rename(columns={
    "EV Segment": "segment",
    "Total No. of EVs supported": "evs_supported"
})
df_supported["source_file"] = "RS_Session_267_AU_580_A.csv"

master_scheme_evs_supported = df_supported[~df_supported["segment"].astype(str).str.contains("Total", case=False, na=False)]
master_scheme_evs_supported.info()
master_scheme_evs_supported

<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Sl. No.        3 non-null      object
 1   segment        3 non-null      object
 2   evs_supported  3 non-null      int64 
 3   source_file    3 non-null      object
dtypes: int64(1), object(3)
memory usage: 120.0+ bytes


,Sl. No.,segment,evs_supported,source_file
0,1,2 wheeler,1428009,RS_Session_267_AU_580_A.csv
1,2,3 wheeler,164180,RS_Session_267_AU_580_A.csv
2,3,4 wheeler,22548,RS_Session_267_AU_580_A.csv


## Insights: Scheme Budget & Subsidy

- Government funding for the EV scheme has scaled up dramatically over time —
  allocation rose from ₹75 crore (2015-16) to ₹5,171.97 crore (2023-24), a
  ~69x increase over 8 years.
- Fund utilization consistently lags allocation, especially in newer years —
  e.g. 2021-22 allocated ₹756.66 crore but an early snapshot shows only
  ₹53.27 crore utilized, suggesting a rollout/implementation lag rather than
  a funding shortage.
- The same financial year's utilization figure changes across successive
  Parliament questions (e.g. 2022-23 utilization reported as ₹1,128.45cr,
  then ₹1,382.90cr, then ₹2,402.51cr in later files) — this reflects
  utilization catching up over time as more of the year's activity gets
  reported, not inconsistent data. For any full-period analysis, the most
  recent snapshot per financial year should be used to avoid triple-counting.
- Subsidy uptake vs. targets shows a significant gap: only 792,529 of a
  targeted 1,000,000 e-2-wheelers were

In [40]:
for f in theme_groups["sales_by_category"]:
    df = load_and_clean(f)
    print(f"\n{f} — shape {df.shape}")
    display(df.head(2))


RS_Session_263_AU_102_A.csv — shape (5, 5)


,Sl. No.,Category,2022,2023,% Growth
0,1,2 Wheelers,631464,859376,36.09
1,2,3 Wheelers,352710,582793,65.23



RS_Session_263_AU_105_C.csv — shape (4, 3)


,Sl. No.,Wheeler Type,Total No. of Vehicle
0,1,2 wheeler,1185829
1,2,3 wheeler,138639



RS_Session_260_AS_241_E.csv — shape (3, 5)


,Category,Fuel Types,Domestic Sales (MoRTH) - 2020-21,Domestic Sales (MoRTH) - 2021-22,Domestic Sales (MoRTH) - 2022-23
0,Passenger Vehicles (e-4W),EVs,5000,19000,47581
1,Three Wheelers (e-3W),EVs,88000,178000,402106


In [41]:
df_growth = load_and_clean("RS_Session_263_AU_102_A.csv")
df_growth = df_growth[~df_growth["Category"].astype(str).str.contains("Total", case=False, na=False)]
df_growth = df_growth.rename(columns={"2022": "sales_2022", "2023": "sales_2023", "% Growth": "pct_growth"})
df_growth["source_file"] = "RS_Session_263_AU_102_A.csv"

master_sales_category_growth = df_growth[["Category", "sales_2022", "sales_2023", "pct_growth", "source_file"]]
master_sales_category_growth.info()
master_sales_category_growth

<class 'pandas.core.frame.DataFrame'>
Index: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Category     4 non-null      object 
 1   sales_2022   4 non-null      int64  
 2   sales_2023   4 non-null      int64  
 3   pct_growth   4 non-null      float64
 4   source_file  4 non-null      object 
dtypes: float64(1), int64(2), object(2)
memory usage: 192.0+ bytes


,Category,sales_2022,sales_2023,pct_growth,source_file
0,2 Wheelers,631464,859376,36.09,RS_Session_263_AU_102_A.csv
1,3 Wheelers,352710,582793,65.23,RS_Session_263_AU_102_A.csv
2,Commercial Vehicles,2649,5673,114.16,RS_Session_263_AU_102_A.csv
3,Passenger Vehicles,38240,82105,114.71,RS_Session_263_AU_102_A.csv


In [42]:
df_total = load_and_clean("RS_Session_263_AU_105_C.csv")
df_total = df_total[~df_total["Wheeler Type"].astype(str).str.contains("Total", case=False, na=False)]
df_total = df_total.rename(columns={"Wheeler Type": "wheeler_type", "Total No. of Vehicle": "total_vehicles"})
df_total["source_file"] = "RS_Session_263_AU_105_C.csv"

master_sales_wheeler_totals = df_total[["wheeler_type", "total_vehicles", "source_file"]]
master_sales_wheeler_totals.info()
master_sales_wheeler_totals

<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   wheeler_type    3 non-null      object
 1   total_vehicles  3 non-null      int64 
 2   source_file     3 non-null      object
dtypes: int64(1), object(2)
memory usage: 96.0+ bytes


,wheeler_type,total_vehicles,source_file
0,2 wheeler,1185829,RS_Session_263_AU_105_C.csv
1,3 wheeler,138639,RS_Session_263_AU_105_C.csv
2,4 wheeler,16991,RS_Session_263_AU_105_C.csv


In [43]:
df_domestic = load_and_clean("RS_Session_260_AS_241_E.csv")

year_cols = [c for c in df_domestic.columns if "Domestic Sales" in c]

master_sales_domestic_by_year = df_domestic.melt(
    id_vars=["Category", "Fuel Types"],
    value_vars=year_cols,
    var_name="year_label",
    value_name="domestic_sales"
)
# extract just the year part, e.g. "2020-21", from the column label
master_sales_domestic_by_year["year"] = master_sales_domestic_by_year["year_label"].str.extract(r'(\d{4}-\d{2})')
master_sales_domestic_by_year = master_sales_domestic_by_year.drop(columns=["year_label"])
master_sales_domestic_by_year["source_file"] = "RS_Session_260_AS_241_E.csv"

master_sales_domestic_by_year.info()
master_sales_domestic_by_year

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Category        9 non-null      object
 1   Fuel Types      9 non-null      object
 2   domestic_sales  9 non-null      int64 
 3   year            9 non-null      object
 4   source_file     9 non-null      object
dtypes: int64(1), object(4)
memory usage: 492.0+ bytes


,Category,Fuel Types,domestic_sales,year,source_file
0,Passenger Vehicles (e-4W),EVs,5000,2020-21,RS_Session_260_AS_241_E.csv
1,Three Wheelers (e-3W),EVs,88000,2020-21,RS_Session_260_AS_241_E.csv
2,Two Wheelers (e-2W),EVs,41000,2020-21,RS_Session_260_AS_241_E.csv
3,Passenger Vehicles (e-4W),EVs,19000,2021-22,RS_Session_260_AS_241_E.csv
4,Three Wheelers (e-3W),EVs,178000,2021-22,RS_Session_260_AS_241_E.csv
5,Two Wheelers (e-2W),EVs,231000,2021-22,RS_Session_260_AS_241_E.csv
6,Passenger Vehicles (e-4W),EVs,47581,2022-23,RS_Session_260_AS_241_E.csv
7,Three Wheelers (e-3W),EVs,402106,2022-23,RS_Session_260_AS_241_E.csv
8,Two Wheelers (e-2W),EVs,728027,2022-23,RS_Session_260_AS_241_E.csv


## Insights: Sales by Vehicle Category

- Domestic EV sales grew dramatically across 2020-21 to 2022-23, but growth
  rates differ sharply by category:
  - Two-wheelers: 41,000 → 231,000 → 728,027 (~18x over 2 years) — by far
    the largest volume and fastest-growing segment.
  - Three-wheelers: 88,000 → 178,000 → 402,106 (~4.6x) — strong, steady
    growth.
  - Passenger vehicles (e-4W): 5,000 → 19,000 → 47,581 (~9.5x) — smallest
    absolute volume, though a comparable multiple of growth.
- 2022→2023 YoY growth % is highest for Commercial Vehicles (114.16%) and
  Passenger Vehicles (114.71%) — but this is misleading in isolation, since
  both categories start from a much smaller base (2,649 and 38,240 units
  respectively) than 2/3-wheelers. High percentage growth on a small base
  is a different story than the sustained high-volume growth seen in
  2/3-wheelers — worth stating both the % and the absolute numbers together
  in the final report to avoid overstating the passenger/commercial trend.

In [45]:
year_cols = ["2019", "2020", "2021", "2022", "2023"]

master_global_ev_share = df_global.melt(
    id_vars=["Region"],
    value_vars=year_cols,
    var_name="year",
    value_name="market_share_pct"
)
master_global_ev_share["source_file"] = "RS_Session_266_AU_552_D_i.csv"

master_global_ev_share.info()
master_global_ev_share

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Region            20 non-null     object 
 1   year              20 non-null     object 
 2   market_share_pct  20 non-null     float64
 3   source_file       20 non-null     object 
dtypes: float64(1), object(3)
memory usage: 772.0+ bytes


,Region,year,market_share_pct,source_file
0,China,2019,93.2,RS_Session_266_AU_552_D_i.csv
1,Europe,2019,2.9,RS_Session_266_AU_552_D_i.csv
2,USA,2019,1.2,RS_Session_266_AU_552_D_i.csv
3,Others,2019,0.6,RS_Session_266_AU_552_D_i.csv
4,China,2020,73.5,RS_Session_266_AU_552_D_i.csv
5,Europe,2020,4.1,RS_Session_266_AU_552_D_i.csv
6,USA,2020,1.2,RS_Session_266_AU_552_D_i.csv
7,Others,2020,4.7,RS_Session_266_AU_552_D_i.csv
8,China,2021,61.2,RS_Session_266_AU_552_D_i.csv
9,Europe,2021,5.7,RS_Session_266_AU_552_D_i.csv


## Insights: Global Comparison

- China dominates global EV market share throughout 2019-2023 but its
  dominance is not steady — it dropped sharply from 93.2% (2019) to a low
  of 61.2% (2021), before recovering to 78.8% (2022) and settling at 68.2%
  (2023). Overall it fell ~25 percentage points across the period even
  with the 2022 rebound.
- Europe shows the clearest sustained growth trend of any region — rising
  from 2.9% (2019) to 18.8% (2023), roughly a 6.5x increase in relative
  share, with growth in every single year of the period.
- USA's share stayed low and largely flat (1.0-2.4%) across all 5 years —
  showing much slower relative EV market growth compared to Europe over
  this period, at least by this market-share measure.
- Overall takeaway: the global EV market is diversifying away from being
  almost entirely China-driven, with Europe the clearest beneficiary of
  that shift; useful context for framing India's own EV growth (covered
  in other themes) against the global backdrop.